In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path(
    "../data/raw/paysim/PS_20174392719_1491204439457_log.csv"
)

print("Dataset exists:", DATA_PATH.exists())
print("Dataset path:", DATA_PATH.resolve())

Dataset exists: True
Dataset path: D:\AI-Projects\RiskPilot-AI\data\raw\paysim\PS_20174392719_1491204439457_log.csv


In [3]:
df_sample = pd.read_csv(
    DATA_PATH,
    nrows=1000
)

print("Sample shape:", df_sample.shape)

Sample shape: (1000, 11)


In [4]:
df_sample.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [5]:
print(df_sample.columns.tolist())

['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']


In [6]:
df_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   step            1000 non-null   int64  
 1   type            1000 non-null   str    
 2   amount          1000 non-null   float64
 3   nameOrig        1000 non-null   str    
 4   oldbalanceOrg   1000 non-null   float64
 5   newbalanceOrig  1000 non-null   float64
 6   nameDest        1000 non-null   str    
 7   oldbalanceDest  1000 non-null   float64
 8   newbalanceDest  1000 non-null   float64
 9   isFraud         1000 non-null   int64  
 10  isFlaggedFraud  1000 non-null   int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 86.1 KB


In [7]:
duplicate_count = df_sample.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


In [8]:
fraud_counts = df_sample["isFraud"].value_counts()

print("Fraud label distribution:")
print(fraud_counts)

Fraud label distribution:
isFraud
0    991
1      9
Name: count, dtype: int64


In [9]:
fraud_percentage = (
    df_sample["isFraud"]
    .value_counts(normalize=True)
    .mul(100)
)

print("Fraud percentage:")
print(fraud_percentage)

Fraud percentage:
isFraud
0    99.1
1     0.9
Name: proportion, dtype: float64


In [10]:
transaction_types = df_sample["type"].value_counts()

print("Transaction type distribution:")
print(transaction_types)

Transaction type distribution:
type
PAYMENT     437
CASH_OUT    230
CASH_IN     183
TRANSFER     99
DEBIT        51
Name: count, dtype: int64


In [11]:
print(df_sample["amount"].describe())

count    1.000000e+03
mean     1.181995e+05
std      2.484799e+05
min      8.730000e+00
25%      4.474727e+03
50%      1.465808e+04
75%      1.427203e+05
max      2.545478e+06
Name: amount, dtype: float64


In [12]:
print("isFlaggedFraud distribution:")
print(df_sample["isFlaggedFraud"].value_counts())

isFlaggedFraud distribution:
isFlaggedFraud
0    1000
Name: count, dtype: int64


In [13]:
# Full dataset analysis using chunks
# This avoids loading the entire 494 MB CSV into memory at once.

total_rows = 0
fraud_count = 0
flagged_fraud_count = 0
missing_count = 0

transaction_type_counts = {}
fraud_by_type = {}

for chunk in pd.read_csv(DATA_PATH, chunksize=200_000):

    total_rows += len(chunk)

    # Fraud counts
    fraud_count += chunk["isFraud"].sum()

    # Flagged fraud counts
    flagged_fraud_count += chunk["isFlaggedFraud"].sum()

    # Missing values
    missing_count += chunk.isnull().sum().sum()

    # Transaction type counts
    type_counts = chunk["type"].value_counts()

    for transaction_type, count in type_counts.items():
        transaction_type_counts[transaction_type] = (
            transaction_type_counts.get(transaction_type, 0) + count
        )

    # Fraud by transaction type
    fraud_table = chunk.groupby("type")["isFraud"].sum()

    for transaction_type, count in fraud_table.items():
        fraud_by_type[transaction_type] = (
            fraud_by_type.get(transaction_type, 0) + count
        )


print("Total transactions:", total_rows)
print("Total fraud transactions:", fraud_count)
print("Total flagged-fraud transactions:", flagged_fraud_count)
print("Total missing values:", missing_count)

print("\nFraud percentage:",
      round((fraud_count / total_rows) * 100, 4))

print("\nTransaction type counts:")
print(transaction_type_counts)

print("\nFraud transactions by type:")
print(fraud_by_type)

Total transactions: 6362620
Total fraud transactions: 8213
Total flagged-fraud transactions: 16
Total missing values: 0

Fraud percentage: 0.1291

Transaction type counts:
{'PAYMENT': 2151495, 'CASH_OUT': 2237500, 'CASH_IN': 1399284, 'TRANSFER': 532909, 'DEBIT': 41432}

Fraud transactions by type:
{'CASH_IN': 0, 'CASH_OUT': 4116, 'DEBIT': 0, 'PAYMENT': 0, 'TRANSFER': 4097}


In [14]:
legitimate_count = total_rows - fraud_count

print("Legitimate transactions:", legitimate_count)
print("Fraud transactions:", fraud_count)

print(
    "Legitimate percentage:",
    round((legitimate_count / total_rows) * 100, 4)
)

print(
    "Fraud percentage:",
    round((fraud_count / total_rows) * 100, 4)
)

Legitimate transactions: 6354407
Fraud transactions: 8213
Legitimate percentage: 99.8709
Fraud percentage: 0.1291


In [17]:
transaction_type_df = pd.DataFrame(
    list(transaction_type_counts.items()),
    columns=["type", "count"]
)

transaction_type_df["percentage"] = (
    transaction_type_df["count"] / total_rows * 100
)

transaction_type_df = transaction_type_df.sort_values(
    "count",
    ascending=False
)

transaction_type_df

,type,count,percentage
1,CASH_OUT,2237500,35.166331
0,PAYMENT,2151495,33.814608
2,CASH_IN,1399284,21.992261
3,TRANSFER,532909,8.375622
4,DEBIT,41432,0.651178


In [18]:
fraud_type_df = pd.DataFrame(
    list(fraud_by_type.items()),
    columns=["type", "fraud_count"]
)

fraud_type_df = fraud_type_df.merge(
    transaction_type_df[["type", "count"]],
    on="type"
)

fraud_type_df["fraud_rate_percent"] = (
    fraud_type_df["fraud_count"]
    / fraud_type_df["count"]
    * 100
)

fraud_type_df.sort_values(
    "fraud_rate_percent",
    ascending=False
)

,type,fraud_count,count,fraud_rate_percent
4,TRANSFER,4097,532909,0.768799
1,CASH_OUT,4116,2237500,0.183955
0,CASH_IN,0,1399284,0.000000
2,DEBIT,0,41432,0.000000
3,PAYMENT,0,2151495,0.000000


In [19]:
amount_count = 0
amount_sum = 0
amount_min = float("inf")
amount_max = float("-inf")

for chunk in pd.read_csv(
    DATA_PATH,
    chunksize=200_000,
    usecols=["amount"]
):

    amounts = chunk["amount"]

    amount_count += amounts.count()
    amount_sum += amounts.sum()

    chunk_min = amounts.min()
    chunk_max = amounts.max()

    if chunk_min < amount_min:
        amount_min = chunk_min

    if chunk_max > amount_max:
        amount_max = chunk_max

amount_mean = amount_sum / amount_count

print("Transaction count:", amount_count)
print("Average amount:", amount_mean)
print("Minimum amount:", amount_min)
print("Maximum amount:", amount_max)

Transaction count: 6362620
Average amount: 179861.90354913071
Minimum amount: 0.0
Maximum amount: 92445516.64


In [20]:
flagged_distribution = {}

for chunk in pd.read_csv(
    DATA_PATH,
    chunksize=200_000,
    usecols=["isFlaggedFraud"]
):

    counts = chunk["isFlaggedFraud"].value_counts()

    for value, count in counts.items():
        flagged_distribution[value] = (
            flagged_distribution.get(value, 0) + count
        )

print(flagged_distribution)

{0: 6362604, 1: 16}
